# PYNQ Latency Benchmark

Thin notebook wrapper around the terminal latency benchmark.

This runs the same `run_pynq_rtt.py` command you would use in the terminal, saves JSON/CSV, and displays the key stats clearly in Jupyter.

Measurement modes:
- `rtt`: direct UDP round-trip time from board to EC2 server and back
- `button_to_visible`: time from a new board button edge to the next returned `PKT_GAME_STATE` being applied and written to BRAM

Trigger modes for `rtt`:
- `auto`: sends probes immediately one after another
- `button`: sends one RTT probe per new board button press

What it does not measure:
- monitor/browser latency
- `rtt` mode does not measure visible movement on screen
- `button_to_visible` mode does not include monitor or browser delay

Most useful stats for the report:
- `p95_rtt_ms` for network stability
- `button_to_visible_p95_ms` for board button to returned server-state visible latency

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

NOTEBOOK_DIR = Path.cwd()
SCRIPT_DIR = NOTEBOOK_DIR if (NOTEBOOK_DIR / "run_pynq_rtt.py").exists() else NOTEBOOK_DIR / "pynq_client_tests"


## Configure Run

In [ ]:
SERVER = "3.9.71.204"
PORT = 9000
LABEL = "idle"
MEASURE = "rtt"  # use "button_to_visible" for button press to returned server-state timing
TRIGGER = "auto"  # in rtt mode, use "button" for one RTT sample per new board button press
SAMPLES = 100
TIMEOUT_S = 1.0
CSV_PATH = Path(f"udp_rtt_{LABEL}.csv")
JSON_PATH = Path(f"udp_rtt_{LABEL}.json")

SERVER, PORT, LABEL, MEASURE, TRIGGER, SAMPLES, TIMEOUT_S, CSV_PATH, JSON_PATH

## Run Benchmark

This runs the same command you would use in the terminal.

In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT_DIR / "run_pynq_rtt.py"),
    "--server", SERVER,
    "--port", str(PORT),
    "--label", LABEL,
    "--measure", MEASURE,
    "--trigger", TRIGGER,
    "--samples", str(SAMPLES),
    "--timeout", str(TIMEOUT_S),
    "--csv-out", str(CSV_PATH),
    "--json-out", str(JSON_PATH),
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

## Load Stats

In [ ]:
report = json.loads(JSON_PATH.read_text())
report

## Key Metrics

In [ ]:
if report.get("measurement") == "button_to_visible":
    stats = report.get("button_to_visible_stats") or {}
    summary = {
        "label": report.get("label"),
        "measurement": report.get("measurement"),
        "trigger": report.get("trigger"),
        "samples_ok": report.get("samples_ok"),
        "samples_lost": report.get("samples_lost"),
        "loss_pct": report.get("loss_pct"),
        "button_to_visible_avg_ms": stats.get("avg_ms"),
        "button_to_visible_p50_ms": stats.get("p50_ms"),
        "button_to_visible_p95_ms": stats.get("p95_ms"),
        "button_to_visible_max_ms": stats.get("max_ms"),
        "button_to_visible_min_ms": stats.get("min_ms"),
        "button_to_visible_stddev_ms": stats.get("stddev_ms"),
    }
else:
    stats = report.get("rtt_stats") or {}
    summary = {
        "label": report.get("label"),
        "measurement": report.get("measurement"),
        "trigger": report.get("trigger"),
        "samples_ok": report.get("samples_ok"),
        "samples_lost": report.get("samples_lost"),
        "loss_pct": report.get("loss_pct"),
        "avg_rtt_ms": stats.get("avg_ms"),
        "p50_rtt_ms": stats.get("p50_ms"),
        "p95_rtt_ms": stats.get("p95_ms"),
        "max_rtt_ms": stats.get("max_ms"),
        "min_rtt_ms": stats.get("min_ms"),
        "stddev_rtt_ms": stats.get("stddev_ms"),
    }
summary

## Reading The Result

- In `rtt` mode, `p95_rtt_ms` is the best headline stability metric for the report.
- In `button_to_visible` mode, `button_to_visible_p95_ms` is the best headline button-to-returned-state latency metric.
- `avg_*_ms` gives the typical latency for the selected measurement mode.
- `max_*_ms` shows the worst spike seen in the run.
- `loss_pct` should ideally stay at `0.0`.